# Retriever Baseline

This notebook is used to test retriever baseline.
- embed query
- dense retrieve
- rerank (bge-reranker-v2-m3)
- in top-k chunks

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))  # từ notebooks/ lên root

load_dotenv()
print("PINECONE_INDEX_NAME:", os.getenv("PINECONE_INDEX_NAME"))
print("PINECONE_NAMESPACE:", os.getenv("PINECONE_NAMESPACE", "default"))

from src.retrieval.retriever import retrieve
from src.retrieval.hybrid_search import hybrid_retrieve

PINECONE_INDEX_NAME: medagnet-rag-vector-db
PINECONE_NAMESPACE: default


c:\STUDY\code\medagent-rag-end2end\.med_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Query1: 
QUERY = "Triệu chứng của bệnh sốt xuất huyết là gì?"

URL_REFERENCE = https://youmed.vn/tin-tuc/sot-xuat-huyet-trieu-chung-cach-dieu-tri-va-nhung-luu-y/

In [3]:
TOP_K = 5
FETCH_K = 20
USE_RERANKER = True

INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

if not INDEX_NAME:
    raise ValueError("Missing PINECONE_INDEX_NAME in .env")

print("Index:", INDEX_NAME)
print("Namespace:", NAMESPACE)
print("Top-k:", TOP_K, "| Fetch-k:", FETCH_K, "| Reranker:", USE_RERANKER)

Index: medagnet-rag-vector-db
Namespace: default
Top-k: 5 | Fetch-k: 20 | Reranker: True


In [5]:
QUERY = "Triệu chứng của bệnh sốt xuất huyết là gì?"

In [4]:
# USE_RERANKER=True
results = retrieve(
    query=QUERY,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
    use_reranker=USE_RERANKER,
)

print(f"Retrieved {len(results)} chunks")
for i, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36363.03it/s]
c:\STUDY\code\medagent-rag-end2end\.med_venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████|

Retrieved 5 chunks
#1 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_2 | score=0.78178215 | rerank=0.9982374906539917
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Triệu chứng sốt xuất huyết | subsection=2. Sốt xuất huyết nặng (hội chứng sốc dengue)
text: Các triệu chứng sốt xuất huyết nặng:1 Giai đoạn đầu của sốt xuất huyết nặng tương tự như sốt xuất huyết và các bệnh sốt siêu vi khác. Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như chảy máu cam, chảy máu ở nướu hoặc dưới da, gây ra vết bầm tím. Tổn thương nặng hơn nữa sẽ gây ra các biến chứng huyết tương thoát khỏi mạch máu, gây chảy chảy máu ồ ạt, sốc (huyết áp thấp).  Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như ch ...
#2 | id=c762c2b8-d55d-438b-9863-2a575ab9d58a_3 | score=0.669307709 | rerank=0.9972648620605469
doc_id=c762c2b8-d55d-438b-9863-2a575ab9d58a | section=Tác dụng của lá đu đủ | subsection=Hỗ trợ điều trị một số triệu chứng bệnh sốt xuất huy

In [9]:
results = retrieve(
    query=QUERY,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
    use_reranker=False,
)

print(f"Retrieved {len(results)} chunks")
for i, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")
    print(item.get("meta_url", ""))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 64016.43it/s]


Retrieved 5 chunks
#1 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_1 | score=0.792964935 | rerank=None
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Triệu chứng sốt xuất huyết | subsection=1. Sốt xuất huyết dạng nhẹ
text: Đây là dạng có biểu hiện các triệu chứng điển hình và không có biến chứng. Biểu hiện đầu tiên của bệnh là sốt và thường kéo dài trong vòng 4 – 7 ngày từ ngày bị truyền virus gây bệnh. Các triệu chứng khác như:1 * Sốt cao, lên đến 40,5°C. * Nhức đầu nghiêm trọng. * Đau phía sau mắt. * Đau khớp và cơ. * Buồn nôn và ói mửa. * Phát ban. Ban thường xuất hiện sau 3 – 4 ngày từ khi bắt đầu sốt. Sau đó, các ban này sẽ thuyên giảm sau 1 – 2 ngày. Bạn có thể bị nổi ban lại một lần nữa vào ngày sau đó. 

#2 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_4 | score=0.788354874 | rerank=None
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Các biểu hiện theo giai đoạn sốt xuất huyết | subsection=1. Giai đoạn sốt2
text: Người bệnh có biểu hiện: * Sốt cao đột ngột 39 – 40°C, sốt

--------------

### QUERY2 = "Những trường hợp không được dùng thuốc Aspirin là gì?"
QUERY2 = "Những trường hợp không được dùng thuốc Aspirin là gì?"

URL_REFERENCE = "https://youmed.vn/tin-tuc/thuoc-aspirin/"

In [6]:
QUERY2 = "Những trường hợp không được dùng thuốc Aspirin là gì?"

In [9]:
results = retrieve(
    query=QUERY2,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
    use_reranker=True,
)

for i, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")
    print(item.get("meta_url", ""))

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3080.83it/s]


#1 | id=ded8943b-e46d-44b5-9e2c-651c7b006daf_9 | score=0.699705064 | rerank=0.9958058595657349
doc_id=ded8943b-e46d-44b5-9e2c-651c7b006daf | section=Đối tượng chống chỉ định dùng Nurofen for children | subsection=Chống chỉ định2
text: Không sử dụng thuốc trong các trường hợp sau: * Dị ứng với bất kỳ thành phần nào của thuốc. * Người có loét dạ dày tiến triển. * Người mẫn cảm vớiaspirinhay với cácthuốc chống viêm không steroidkhác. Những người mẫn cảm như: bệnh nhân hen, viêm mũi, nổi mày đay sau khi dùng aspirin. * Người bị hen hay bị co thắt phế quản,bệnh tim mạch, rối loạn chảy máu, tiền căn loét dạ dày tá tràng, suy thận (lưu lượng lọc cầu thận (GFR) < 30 ml/phút) hoặc suy gan. * Người đang được điều trị bằng thuốc chống đ ...

#2 | id=fcba10fd-ca60-410f-bbc5-14f52e9b4b23_9 | score=0.730354249 | rerank=0.9938042163848877
doc_id=fcba10fd-ca60-410f-bbc5-14f52e9b4b23 | section=Chống chỉ định của thuốc Aspirin STELLA | subsection=
text: Nhóm bệnh nhân sau không được sử dụng Aspirin STEL

In [10]:
results = retrieve(
    query=QUERY2,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
    use_reranker=False,
)

for i, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")
    print(item.get("meta_url", ""))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6436.92it/s]


#1 | id=fcba10fd-ca60-410f-bbc5-14f52e9b4b23_9 | score=0.730354249 | rerank=None
doc_id=fcba10fd-ca60-410f-bbc5-14f52e9b4b23 | section=Chống chỉ định của thuốc Aspirin STELLA | subsection=
text: Nhóm bệnh nhân sau không được sử dụng Aspirin STELLA:1 2 * Bệnh nhân có tiền sử dị ứng với bất kỳ thành phần của thuốc. * Bệnh nhân có triệu chứng của bệnhhen, viêm mũi hoặc mày đay hoặc có tiền sử bị hen. * Bệnh nhân mắc bệnh lý giảm tiểu cầu. * Bệnh nhân đang bị có triệu chứng của loét dạ dày hoặc tá tràng. * Người đang bị xuất huyết tiêu hóa, hoặc đang bị chảy máu do nguyên nhân khác. * Bệnh nhânsuy timvừa và nặng,suy gan, thận nặng. * Phụ nữ có thai đang trong 3 tháng cuối thai kỳ. 

#2 | id=9d2d173e-c545-421c-b7c1-09d266d083d6_3 | score=0.722503603 | rerank=None
doc_id=9d2d173e-c545-421c-b7c1-09d266d083d6 | section=Trường hợp không dùng thuốc Aspirin | subsection=
text: Dị ứng với Aspirin hoặc dị ứng với bất kì các thành phần nào có trong thuốc. Có triệu chứng hen, bị viêm mũi hoặc nổi mày

## Hybrid_Search

In [8]:
hybrid_results = hybrid_retrieve(
    query=QUERY,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
)

for i, item in enumerate(hybrid_results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")
    print(item.get("meta_url", ""))

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5554.34it/s]


#1 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_2 | score=0.78178215 | rerank=0.9982374906539917
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Triệu chứng sốt xuất huyết | subsection=2. Sốt xuất huyết nặng (hội chứng sốc dengue)
text: Các triệu chứng sốt xuất huyết nặng:1 Giai đoạn đầu của sốt xuất huyết nặng tương tự như sốt xuất huyết và các bệnh sốt siêu vi khác. Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như chảy máu cam, chảy máu ở nướu hoặc dưới da, gây ra vết bầm tím. Tổn thương nặng hơn nữa sẽ gây ra các biến chứng huyết tương thoát khỏi mạch máu, gây chảy chảy máu ồ ạt, sốc (huyết áp thấp).  Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như ch ...

#2 | id=c762c2b8-d55d-438b-9863-2a575ab9d58a_3 | score=0.669307709 | rerank=0.9972648620605469
doc_id=c762c2b8-d55d-438b-9863-2a575ab9d58a | section=Tác dụng của lá đu đủ | subsection=Hỗ trợ điều trị một số triệu chứng bệnh sốt xuất huyết
text: Sốt xuất 

In [9]:
hybrid_results = hybrid_retrieve(
    query=QUERY,
    top_k=TOP_K,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    fetch_k=FETCH_K,
)

for i, item in enumerate(hybrid_results, start=1):
    print("=" * 80)
    print(f"#{i} | id={item.get('id')} | score={item.get('score')} | rerank={item.get('rerank_score')}")
    print(f"doc_id={item.get('doc_id')} | section={item.get('section')} | subsection={item.get('subsection')}")
    text = (item.get("text") or "").strip().replace("\n", " ")
    print("text:", text[:500], "..." if len(text) > 500 else "")
    print(item.get("meta_url", ""))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36587.75it/s]


#1 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_2 | score=0.78178215 | rerank=None
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Triệu chứng sốt xuất huyết | subsection=2. Sốt xuất huyết nặng (hội chứng sốc dengue)
text: Các triệu chứng sốt xuất huyết nặng:1 Giai đoạn đầu của sốt xuất huyết nặng tương tự như sốt xuất huyết và các bệnh sốt siêu vi khác. Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như chảy máu cam, chảy máu ở nướu hoặc dưới da, gây ra vết bầm tím. Tổn thương nặng hơn nữa sẽ gây ra các biến chứng huyết tương thoát khỏi mạch máu, gây chảy chảy máu ồ ạt, sốc (huyết áp thấp).  Nhưng kèm theo đó là các tổn thương nghiêm trọng như mạch máu và mạch bạch huyết, như ch ...

#2 | id=b5613ad3-5b3d-41be-95ab-dd5e3632d066_1 | score=0.792964935 | rerank=None
doc_id=b5613ad3-5b3d-41be-95ab-dd5e3632d066 | section=Triệu chứng sốt xuất huyết | subsection=1. Sốt xuất huyết dạng nhẹ
text: Đây là dạng có biểu hiện các triệu chứng điển hình và không 